In [18]:
!pip install -q numpy huggingface_hub safetensors


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import numpy as np
from huggingface_hub import hf_hub_download
from safetensors.numpy import load_file

REPO = "nilc-nlp/word2vec-skip-gram-300d"

vectors = load_file(hf_hub_download(REPO, "embeddings.safetensors"))["embeddings"].astype(np.float32)
with open(hf_hub_download(REPO, "vocab.txt"), encoding="utf-8") as f:
    vocab = [w.strip() for w in f]

vectors /= np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12


class Word2Vec:
    def __init__(self, vocab, vectors):
        self.vocab = vocab
        self.vectors = vectors
        self.idx = {w: i for i, w in enumerate(vocab)}

    def __contains__(self, palavra):
        return palavra in self.idx

    def similarity(self, a, b):
        return float(self.vectors[self.idx[a]] @ self.vectors[self.idx[b]])

    def most_similar(self, palavra, topn=10):
        scores = self.vectors @ self.vectors[self.idx[palavra]]
        top = np.argsort(-scores)[1:topn + 1]
        return [(self.vocab[i], float(scores[i])) for i in top]


model = Word2Vec(vocab, vectors)

c:\Users\hique\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hique\.cache\huggingface\hub\models--nilc-nlp--word2vec-skip-gram-300d. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [20]:
print(model.similarity("cachorro", "cão"))
print(model.similarity("cachorro", "gato"))
print(model.similarity("cachorro", "foguete"))

0.754359781742096
0.645607054233551
0.1518494039773941


In [21]:
LIMIAR = 0.4

def ponte_valida(caminho):
    for a, b in zip(caminho, caminho[1:]):
        s = model.similarity(a, b)
        print(f"{a} -> {b}: {s:.2f}")
        if s < LIMIAR:
            return False
    return True

ponte_valida(["cachorro", "animal", "espaço", "foguete"])

cachorro -> animal: 0.39


False

In [ ]:
class Palavra:

    def __init__(self, palavra):
        self.palavra = palavra
        self.conectadas = set()

In [ ]:
LIMIAR = 0.4


class Palavra:
    def __init__(self, palavra, pai=None, lado=None):
        self.palavra = palavra
        self.pai = pai    # palavra à qual esta se conectou
        self.lado = lado  # "inicio" ou "fim"


def caminho(no):
    """Retorna a lista da raiz do lado até esta palavra."""
    passos = []
    while no is not None:
        passos.append(no.palavra)
        no = no.pai
    return list(reversed(passos))


def mostrar_ponte(lista):
    texto = lista[0]
    for a, b in zip(lista, lista[1:]):
        texto += f" --({model.similarity(a, b):.2f})--> {b}"
    print(texto)


def mostrar_lados():
    for lado in ("inicio", "fim"):
        ramos = [caminho(p) for p in palavras if p.lado == lado]
        ponta = max(ramos, key=len)
        print(f"  lado {lado}: {' -> '.join(ponta)}")


inicio = Palavra("amor", lado="inicio")
fim = Palavra("dinheiro", lado="fim")

assert inicio.palavra in model and fim.palavra in model, "Palavra fora do vocabulário"

palavras = [inicio, fim]
print(f"Construa uma ponte entre '{inicio.palavra}' e '{fim.palavra}'. Limiar: {LIMIAR}")
print(f"Similaridade direta: {model.similarity(inicio.palavra, fim.palavra):.2f}\n")

while True:
    nova = input("Digite uma palavra (ou 'sair'): ").strip().lower()

    if not nova:
        continue
    if nova == "sair":
        print("Jogo encerrado.")
        break
    if nova not in model:
        print(f"'{nova}' não está no vocabulário.\n")
        continue
    if any(p.palavra == nova for p in palavras):
        print(f"'{nova}' já foi usada.\n")
        continue

    lado_ini = [p for p in palavras if p.lado == "inicio"]
    lado_fim = [p for p in palavras if p.lado == "fim"]

    m_ini = max(lado_ini, key=lambda p: model.similarity(nova, p.palavra))
    m_fim = max(lado_fim, key=lambda p: model.similarity(nova, p.palavra))
    s_ini = model.similarity(nova, m_ini.palavra)
    s_fim = model.similarity(nova, m_fim.palavra)

    # vitória: a palavra liga os dois lados
    if s_ini >= LIMIAR and s_fim >= LIMIAR:
        ponte = caminho(m_ini) + [nova] + list(reversed(caminho(m_fim)))
        print(f"\nPonte completa em {len(ponte) - 2} palavras!")
        mostrar_ponte(ponte)
        break

    if s_ini < LIMIAR and s_fim < LIMIAR:
        print(f"'{nova}' não se conecta. "
              f"Lado início: '{m_ini.palavra}' ({s_ini:.2f}) | "
              f"Lado fim: '{m_fim.palavra}' ({s_fim:.2f})\n")
        continue

    # conecta no lado mais próximo
    if s_ini >= s_fim:
        palavras.append(Palavra(nova, pai=m_ini, lado="inicio"))
        print(f"'{nova}' conectou em '{m_ini.palavra}' ({s_ini:.2f})")
    else:
        palavras.append(Palavra(nova, pai=m_fim, lado="fim"))
        print(f"'{nova}' conectou em '{m_fim.palavra}' ({s_fim:.2f})")

    mostrar_lados()
    print()

Construa uma ponte entre 'amor' e 'dinheiro'. Limiar: 0.4
Similaridade direta: 0.09

'amor' já foi usada.

'casal' não se conecta. Lado início: 'amor' (0.27) | Lado fim: 'dinheiro' (0.17)

'casamento' não se conecta. Lado início: 'amor' (0.39) | Lado fim: 'dinheiro' (0.07)

'paixao' não se conecta. Lado início: 'amor' (0.15) | Lado fim: 'dinheiro' (0.05)

'apaixonar' não se conecta. Lado início: 'amor' (0.29) | Lado fim: 'dinheiro' (0.01)

'divorcio' não se conecta. Lado início: 'amor' (0.25) | Lado fim: 'dinheiro' (0.14)

'beijo' conectou em 'amor' (0.50)
  lado inicio: amor -> beijo
  lado fim: dinheiro

'casal' não se conecta. Lado início: 'beijo' (0.28) | Lado fim: 'dinheiro' (0.17)

'casamento' não se conecta. Lado início: 'amor' (0.39) | Lado fim: 'dinheiro' (0.07)

'marido' não se conecta. Lado início: 'amor' (0.38) | Lado fim: 'dinheiro' (0.14)

'mulher' não se conecta. Lado início: 'amor' (0.26) | Lado fim: 'dinheiro' (0.01)

'grana' conectou em 'dinheiro' (0.47)
  lado inicio

: 